# 📊 Dashboard KPI Interactif — NexaCommerce Cameroun
## Visualisation Plotly — Livrable L4

Ce notebook produit un **dashboard multi-KPI interactif** couvrant quatre angles métier :
- **Volume & Croissance** — évolution temporelle des commandes et du CA
- **Qualité opérationnelle** — taux de retard, délais, statuts
- **Géographie** — comparaison des trois villes
- **Produits & Livreurs** — CA par catégorie, performance individuelle

> Toutes les cellules sont autonomes après l'exécution de la cellule de setup (Section 0).

---
## 0. Setup — Imports, Données, Normalisation

In [2]:
import re
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Palette NexaCommerce ──────────────────────────────────────
NAVY    = "#1E3A5F"
GOLD    = "#C8973A"
WHITE   = "#FFFFFF"
SUCCESS = "#15803D"
DANGER  = "#B91C1C"
AMBER   = "#B45309"
LIGHT   = "#F1F5F9"
GRAY    = "#64748B"
C_CITY  = {"Douala": "#2563EB", "Yaoundé": "#16A34A", "Bafoussam": "#DC2626"}

# ── Chargement ────────────────────────────────────────────────
DATA_DIR    = Path("../data")
orders      = pd.read_csv(DATA_DIR / "orders.csv",    low_memory=False)
customers   = pd.read_csv(DATA_DIR / "customers.csv", low_memory=False)
couriers_df = pd.read_csv(DATA_DIR / "couriers.csv")

# ── Normalisation ─────────────────────────────────────────────
STATUS_MAP = [(r"^livr","delivered"),(r"retard","late"),
              (r"annul","cancelled"),(r"cours","in_progress")]

def norm_status(s):
    if pd.isna(s): return None
    s = str(s).strip().lower()
    for pat, val in STATUS_MAP:
        if re.search(pat, s): return val
    return "unknown"

def norm_city(c):
    if pd.isna(c): return None
    c = str(c).strip().lower()
    if "douala" in c:    return "Douala"
    if "yaound" in c:    return "Yaoundé"
    if "bafoussam" in c: return "Bafoussam"
    return "Autre"

CAT_MAP = [
    (r"restaur|resatur","Restauration"),(r"boisson","Boissons"),
    (r"epicer|épic","Épicerie"),(r"produit|frais","Produits frais"),
    (r"pharmac","Pharmacie"),(r"electron","Électronique"),
    (r"cosmet","Cosmétiques"),(r"hygien","Hygiène"),
    (r"vetement|vêtement","Vêtements"),(r"papeter","Papeterie"),
]

def norm_cat(x):
    if pd.isna(x): return None
    x = str(x).strip().lower()
    for pat, val in CAT_MAP:
        if re.search(pat, x): return val
    return None

DATE_FMTS = ["%Y-%m-%d %H:%M:%S","%Y/%m/%d %H:%M","%d/%m/%Y %H:%M",
             "%Y-%m-%d","%d-%m-%Y","%Y/%m/%d"]

def parse_date(d):
    if pd.isna(d): return pd.NaT
    for f in DATE_FMTS:
        try: return pd.to_datetime(d, format=f)
        except: continue
    return pd.NaT

orders["status_clean"] = orders["status"].apply(norm_status)
orders["city_clean"]   = orders["city"].apply(norm_city)
orders["cat_clean"]    = orders["product_category"].apply(norm_cat)
orders["date_clean"]   = orders["order_date"].apply(parse_date)
orders["ym"]           = orders["date_clean"].dt.to_period("M").astype(str)
orders["year"]         = orders["date_clean"].dt.year

# Dataset principal : montants valides, jusqu'à mars 2025
ov = orders[(orders["total_amount_xaf"] > 0) & (orders["ym"] <= "2025-03")].copy()

print(f"✅ Dataset prêt : {len(ov):,} commandes valides")
print(f"   Période      : {ov['ym'].min()}  →  {ov['ym'].max()}")
print(f"   CA total     : {ov['total_amount_xaf'].sum()/1e6:.1f} M XAF")
print(f"   Taux retard  : {(ov['status_clean']=='late').mean()*100:.1f} %")

✅ Dataset prêt : 10,481 commandes valides
   Période      : 2022-01  →  2025-03
   CA total     : 348.4 M XAF
   Taux retard  : 22.6 %


---
## 1. Scorecard — KPIs Globaux

Six indicateurs clés en un coup d'œil.

In [3]:
# ── Calcul des KPIs ──────────────────────────────────────────
n_orders   = len(ov)
ca_total   = ov["total_amount_xaf"].sum()
panier_moy = ov["total_amount_xaf"].mean()
late_rate  = (ov["status_clean"] == "late").mean() * 100
deliv_rate = (ov["status_clean"] == "delivered").mean() * 100
del_moy    = ov[ov["delivery_time_min"] > 0]["delivery_time_min"].mean()

# ── Figure indicator (gauge) ─────────────────────────────────
fig = go.Figure()

kpis = [
    {"value": n_orders,           "title": "Commandes analysées",  "suffix": "",       "color": NAVY,    "col": 1},
    {"value": ca_total / 1e6,     "title": "CA Total (M XAF)",     "suffix": " M",     "color": GOLD,    "col": 2},
    {"value": panier_moy,         "title": "Panier Moyen (XAF)",   "suffix": " XAF",   "color": "#0369A1","col": 3},
    {"value": late_rate,          "title": "Taux de Retard",        "suffix": " %",     "color": DANGER,  "col": 4},
    {"value": deliv_rate,         "title": "Taux de Livraison",     "suffix": " %",     "color": SUCCESS, "col": 5},
    {"value": del_moy,            "title": "Délai Moyen (min)",     "suffix": " min",   "color": AMBER,   "col": 6},
]

for k in kpis:
    fmt = f"{k['value']:,.0f}{k['suffix']}" if k["suffix"] != " M" else f"{k['value']:.1f}{k['suffix']}"
    fig.add_trace(go.Indicator(
        mode="number",
        value=k["value"],
        number={
            "valueformat": ",.0f" if k["suffix"] not in [" %", " min", " M"] else ".1f",
            "suffix": k["suffix"],
            "font": {"size": 42, "color": k["color"], "family": "Georgia"},
        },
        title={"text": f"<b>{k['title']}</b>", "font": {"size": 13, "color": GRAY}},
        domain={"column": k["col"] - 1, "row": 0},
    ))

fig.update_layout(
    grid={"rows": 1, "columns": 6},
    height=180,
    margin=dict(t=40, b=10, l=20, r=20),
    paper_bgcolor=WHITE,
    title=dict(
        text="<b>NexaCommerce Cameroun — KPIs Globaux</b>  ·  Jan 2022 – Mar 2025",
        font=dict(size=16, color=NAVY, family="Georgia"),
        x=0.5,
    ),
)
fig.show()

---
## 2. Évolution Temporelle — Volume & CA

Graphique à deux axes : barres pour le volume de commandes, ligne pour le CA mensuel.
Les zones colorées marquent les années.

In [ ]:
monthly = (
    ov.groupby("ym")
    .agg(orders=("order_id","count"), revenue=("total_amount_xaf","sum"))
    .reset_index()
)
monthly["revenue_M"] = monthly["revenue"] / 1e6

fig = make_subplots(
    specs=[[{"secondary_y": True}]],
    figure=go.Figure()
)

# Barres volume
fig.add_trace(go.Bar(
    x=monthly["ym"],
    y=monthly["orders"],
    name="Volume commandes",
    marker_color=NAVY,
    marker_opacity=0.75,
    hovertemplate="<b>%{x}</b><br>Commandes : %{y:,}<extra></extra>",/ra
), secondary_y=False)

# Ligne CA
fig.add_trace(go.Scatter(
    x=monthly["ym"],
    y=monthly["revenue_M"],
    name="CA mensuel (M XAF)",
    mode="lines+markers",
    line=dict(color=GOLD, width=2.5),
    marker=dict(size=5, color=GOLD),
    hovertemplate="<b>%{x}</b><br>CA : %{y:.1f} M XAF<extra></extra>",
), secondary_y=True)

# Annotations années
for yr, label, xref in [("2022-06","2022","x"),("2023-06","2023","x"),
                          ("2024-06","2024","x"),("2025-01","2025","x")]:
    fig.add_annotation(
        x=yr, y=1.0, yref="paper",
        text=f"<b>{label}</b>", showarrow=False,
        font=dict(size=11, color=GRAY), yanchor="bottom"
    )

# Annotation croissance
fig.add_annotation(
    x="2024-09", y=ov[ov["ym"]<="2024-12"].groupby("ym")["order_id"].count().max() * 0.9,
    text="<b>+211 %</b><br>2022 → 2024",
    showarrow=True, arrowhead=2, arrowcolor=GOLD, arrowwidth=1.5,
    bgcolor="#FEF3C7", bordercolor=GOLD, borderwidth=1,
    font=dict(size=11, color=AMBER),
    ax=-60, ay=-40,
)

fig.update_layout(
    title=dict(text="<b>Évolution mensuelle — Volume de commandes & Chiffre d'affaires</b>",
               font=dict(size=16, color=NAVY, family="Georgia"), x=0.5),
    height=440,
    paper_bgcolor=WHITE,
    plot_bgcolor="#FAFAFA",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center"),
    hovermode="x unified",
    xaxis=dict(tickangle=-45, tickfont=dict(size=9), showgrid=False),
    margin=dict(t=80, b=60, l=60, r=60),
)
fig.update_yaxes(title_text="Commandes / mois", secondary_y=False,
                 showgrid=True, gridcolor="#E2E8F0", zeroline=False)
fig.update_yaxes(title_text="CA (Millions XAF)", secondary_y=True,
                 showgrid=False, zeroline=False)

fig.show()

---
## 3. Taux de Retard — Évolution & Zone Critique

Ligne du taux de retard mensuel avec bande de tolérance.
Zone rouge = dépassement du seuil cible de 20 %.

In [5]:
monthly_late = (
    ov.groupby("ym")["status_clean"]
    .apply(lambda x: (x == "late").mean() * 100)
    .reset_index()
    .rename(columns={"status_clean": "late_rate"})
)

TARGET = 20.0  # seuil cible

fig = go.Figure()

# Zone rouge au-dessus du seuil
fig.add_hrect(
    y0=TARGET, y1=monthly_late["late_rate"].max() + 2,
    fillcolor=DANGER, opacity=0.06,
    layer="below", line_width=0,
    annotation_text="Zone critique (> 20 %)",
    annotation_position="top right",
    annotation_font=dict(color=DANGER, size=11),
)

# Ligne cible
fig.add_hline(
    y=TARGET, line_dash="dash", line_color=DANGER,
    line_width=1.5,
    annotation_text=f"Cible : {TARGET} %",
    annotation_position="bottom right",
    annotation_font=dict(color=DANGER, size=11),
)

# Aire remplie
fig.add_trace(go.Scatter(
    x=monthly_late["ym"],
    y=monthly_late["late_rate"],
    fill="tozeroy",
    fillcolor="rgba(30, 58, 95, 0.12)",
    line=dict(color=NAVY, width=2.5),
    mode="lines+markers",
    marker=dict(
        size=7,
        color=[DANGER if v > TARGET else SUCCESS for v in monthly_late["late_rate"]],
        line=dict(width=1.5, color=WHITE),
    ),
    name="Taux retard mensuel",
    hovertemplate="<b>%{x}</b><br>Taux retard : %{y:.1f} %<extra></extra>",
))

# Moyenne mobile 3 mois
monthly_late["ma3"] = monthly_late["late_rate"].rolling(3, center=True).mean()
fig.add_trace(go.Scatter(
    x=monthly_late["ym"],
    y=monthly_late["ma3"],
    mode="lines",
    line=dict(color=GOLD, width=2, dash="dot"),
    name="Moy. mobile 3 mois",
    hovertemplate="<b>%{x}</b><br>Moy. mobile : %{y:.1f} %<extra></extra>",
))

fig.update_layout(
    title=dict(
        text="<b>Taux de retard mensuel</b>  ·  Cible : < 20 %  ·  Actuel : 22,6 %",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    height=400,
    paper_bgcolor=WHITE,
    plot_bgcolor="#FAFAFA",
    xaxis=dict(tickangle=-45, tickfont=dict(size=9), showgrid=False),
    yaxis=dict(title="Taux de retard (%)", showgrid=True,
               gridcolor="#E2E8F0", zeroline=False, range=[0, 35]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center"),
    hovermode="x unified",
    margin=dict(t=80, b=60, l=60, r=60),
)
fig.show()

---
## 4. Performances Géographiques — Radar & Comparaison

Radar chart : chaque axe est un KPI normalisé (0–1).
Permet de comparer les villes sur plusieurs dimensions simultanément.

In [13]:
city_stats = (
    ov.groupby("city_clean")
    .agg(
        orders       = ("order_id",          "count"),
        revenue      = ("total_amount_xaf",  "sum"),
        avg_basket   = ("total_amount_xaf",  "mean"),
        late_rate    = ("status_clean",      lambda x: (x == "late").mean() * 100),
        avg_delivery = ("delivery_time_min", lambda x: x[x > 0].mean()),
    )
    .loc[["Douala", "Yaoundé", "Bafoussam"]]
    .round(1)
    .reset_index()
)

# Normalisation 0-1 (1 = meilleur, donc on inverse late_rate et avg_delivery)
def norm01(series, invert=False):
    mn, mx = series.min(), series.max()
    n = (series - mn) / (mx - mn) if mx != mn else series * 0 + 0.5
    return 1 - n if invert else n

categories = ["Volume", "CA", "Panier Moy.", "Ponctualité", "Rapidité"]

radar_data = pd.DataFrame({
    "Volume":      norm01(city_stats["orders"]),
    "CA":          norm01(city_stats["revenue"]),
    "Panier Moy.": norm01(city_stats["avg_basket"]),
    "Ponctualité": norm01(city_stats["late_rate"], invert=True),
    "Rapidité":    norm01(city_stats["avg_delivery"], invert=True),
})

fig = go.Figure()

for i, row in city_stats.iterrows():
    city = row["city_clean"]
    vals = radar_data.iloc[i].tolist()
    vals_closed = vals + [vals[0]]
    cats_closed = categories + [categories[0]]

    fig.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=cats_closed,
        fill="toself",
        fillcolor=C_CITY.get(city, GRAY),
        opacity=0.2,
        line=dict(color=C_CITY.get(city, GRAY), width=2.5),
        name=city,
        hovertemplate=(
            f"<b>{city}</b><br>"
            f"Volume : {row['orders']:,} cmd<br>"
            f"CA : {row['revenue']/1e6:.1f} M XAF<br>"
            f"Panier moy. : {row['avg_basket']:,.0f} XAF<br>"
            f"Taux retard : {row['late_rate']:.1f} %<br>"
            f"Délai moy. : {row['avg_delivery']:.0f} min<extra></extra>"
        ),
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True, range=[0, 1],
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            ticktext=["0", "", "0.5", "", "1"],
            tickfont=dict(size=9, color=GRAY),
            gridcolor="#E2E8F0",
        ),
        angularaxis=dict(tickfont=dict(size=12, color=NAVY)),
        bgcolor="#FAFAFA",
    ),
    title=dict(
        text="<b>Radar de Performance par Ville</b>  ·  Score normalisé 0–1 (1 = meilleur)",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, x=0.5, xanchor="center"),
    height=480,
    paper_bgcolor=WHITE,
    margin=dict(t=80, b=80, l=60, r=60),
)
fig.show()

---
## 5. Heatmap Retards — Jour × Heure

Les zones rouges = créneaux à risque opérationnel.
Outil de décision direct pour le planning livreurs.

In [7]:
DOW_ORDER  = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
DOW_LABELS = ["Lun","Mar","Mer","Jeu","Ven","Sam","Dim"]

hm_data = (
    ov.groupby(["order_day_of_week", "order_hour"])["status_clean"]
    .apply(lambda x: round((x == "late").mean() * 100, 1))
    .reset_index()
    .rename(columns={"status_clean": "late_rate"})
)

pivot = (
    hm_data
    .pivot(index="order_day_of_week", columns="order_hour", values="late_rate")
    .reindex([d for d in DOW_ORDER if d in hm_data["order_day_of_week"].unique()])
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[f"{h}h" for h in pivot.columns],
    y=DOW_LABELS[:len(pivot)],
    colorscale=[
        [0.0,  "#EFF6FF"],
        [0.3,  "#BFDBFE"],
        [0.55, "#FDE68A"],
        [0.75, "#FCA5A5"],
        [1.0,  "#991B1B"],
    ],
    zmin=0, zmax=40,
    text=[[f"{v:.0f}%" if not np.isnan(v) else "" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    textfont=dict(size=9, color="black"),
    hovertemplate="<b>%{y} — %{x}</b><br>Taux retard : %{z:.1f} %<extra></extra>",
    colorbar=dict(
        title=dict(text="Retard %", font=dict(size=11, color=GRAY)),
        tickfont=dict(size=10, color=GRAY),
        len=0.8,
    ),
))

# Annotations spikes
for h, label in [(7, "Rush 7h"), (18, "Rush soir")]:
    fig.add_annotation(
        x=f"{h}h", y=0, yref="paper",
        text=f"⚠ {label}", showarrow=False,
        font=dict(size=10, color=AMBER),
        yanchor="bottom",
    )

fig.update_layout(
    title=dict(
        text="<b>Heatmap — Taux de retard par Jour & Heure</b>"
             "  ·  Rouge = créneau critique",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    height=360,
    paper_bgcolor=WHITE,
    xaxis=dict(tickfont=dict(size=10), title="Heure de la commande"),
    yaxis=dict(tickfont=dict(size=11), title=""),
    margin=dict(t=80, b=60, l=60, r=80),
)
fig.show()

---
## 6. Catégories Produits — CA vs Volume vs Panier

Scatter bubble : chaque bulle = une catégorie.
Axe X = volume commandes, Axe Y = panier moyen, Taille = CA total.

In [8]:
cat_stats = (
    ov[ov["cat_clean"].notna()]
    .groupby("cat_clean")
    .agg(
        orders     = ("order_id",         "count"),
        revenue    = ("total_amount_xaf", "sum"),
        avg_basket = ("total_amount_xaf", "mean"),
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)
cat_stats["revenue_M"]   = cat_stats["revenue"] / 1e6
cat_stats["avg_basket_k"] = cat_stats["avg_basket"] / 1000

COLORS_CAT = px.colors.qualitative.Set2[:len(cat_stats)]

fig = go.Figure()

for i, row in cat_stats.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["orders"]],
        y=[row["avg_basket_k"]],
        mode="markers+text",
        marker=dict(
            size=max(row["revenue_M"] * 3.5, 15),
            color=COLORS_CAT[i % len(COLORS_CAT)],
            opacity=0.8,
            line=dict(width=1.5, color=WHITE),
        ),
        text=[row["cat_clean"]],
        textposition="top center",
        textfont=dict(size=10, color=NAVY),
        name=row["cat_clean"],
        hovertemplate=(
            f"<b>{row['cat_clean']}</b><br>"
            f"Commandes : {row['orders']:,}<br>"
            f"CA : {row['revenue_M']:.1f} M XAF<br>"
            f"Panier moyen : {row['avg_basket']:,.0f} XAF<extra></extra>"
        ),
    ))

# Quadrants
mid_x = cat_stats["orders"].median()
mid_y = cat_stats["avg_basket_k"].median()
fig.add_vline(x=mid_x, line_dash="dot", line_color=GRAY, line_width=1,
              annotation_text="Volume médian", annotation_position="top",
              annotation_font=dict(size=9, color=GRAY))
fig.add_hline(y=mid_y, line_dash="dot", line_color=GRAY, line_width=1,
              annotation_text="Panier médian", annotation_position="right",
              annotation_font=dict(size=9, color=GRAY))

fig.update_layout(
    title=dict(
        text="<b>Catégories Produits</b>  ·  Volume vs Panier Moyen  ·  Taille = CA total",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    height=480,
    paper_bgcolor=WHITE,
    plot_bgcolor="#FAFAFA",
    xaxis=dict(title="Nombre de commandes", showgrid=True,
               gridcolor="#E2E8F0", zeroline=False),
    yaxis=dict(title="Panier moyen (milliers XAF)", showgrid=True,
               gridcolor="#E2E8F0", zeroline=False),
    showlegend=False,
    margin=dict(t=80, b=60, l=80, r=60),
)
fig.show()

---
## 7. Performance Livreurs — Note vs Taux de Retard

Scatter : axe X = note client, axe Y = taux de retard.
Le quadrant haut-droit (bonne note + fort retard) = **anomalie RH**.

In [10]:
cp = ov.merge(couriers_df[["courier_id","name","rating"]], on="courier_id", how="left")

courier_perf = (
    cp.groupby(["courier_id","name"])
    .agg(
        orders       = ("order_id",          "count"),
        late_rate    = ("status_clean",      lambda x: (x == "late").mean() * 100),
        avg_delivery = ("delivery_time_min", "mean"),
        rating       = ("rating",            "first"),
    )
    .reset_index()
    .dropna(subset=["rating"])
    .round(1)
)

# Couleur selon risque
def risk_color(row):
    if row["late_rate"] > 27 and row["rating"] >= 3.5:
        return DANGER    # anomalie : retard élevé + bonne note
    elif row["late_rate"] > 25:
        return AMBER     # retard élevé
    else:
        return SUCCESS   # normal

courier_perf["color"] = courier_perf.apply(risk_color, axis=1)

fig = go.Figure()

for _, row in courier_perf.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["rating"]],
        y=[row["late_rate"]],
        mode="markers",
        marker=dict(
            size=max(row["orders"] * 0.25, 10),
            color=row["color"],
            opacity=0.75,
            line=dict(width=1.5, color=WHITE),
        ),
        name=row["name"],
        hovertemplate=(
            f"<b>{row['name']}</b><br>"
            f"Commandes : {row['orders']:.0f}<br>"
            f"Taux retard : {row['late_rate']:.1f} %<br>"
            f"Délai moy. : {row['avg_delivery']:.0f} min<br>"
            f"Note : {row['rating']:.1f}/5<extra></extra>"
        ),
    ))

# Annoter les anomalies (retard > 27% et note >= 3.5)
anomalies = courier_perf[(courier_perf["late_rate"] > 27) & (courier_perf["rating"] >= 3.5)]
for _, row in anomalies.iterrows():
    fig.add_annotation(
        x=row["rating"], y=row["late_rate"],
        text=f"⚠ {row['name'].split()[0]}",
        showarrow=True, arrowhead=2, arrowcolor=DANGER,
        bgcolor="#FEF2F2", bordercolor=DANGER, borderwidth=1,
        font=dict(size=10, color=DANGER),
        ax=40, ay=-30,
    )

# Lignes de référence
avg_late = courier_perf["late_rate"].mean()
fig.add_hline(y=avg_late, line_dash="dash", line_color=GRAY, line_width=1.2,
              annotation_text=f"Moy. {avg_late:.1f} %",
              annotation_position="right",
              annotation_font=dict(size=10, color=GRAY))
fig.add_vline(x=3.0, line_dash="dash", line_color=GRAY, line_width=1.2,
              annotation_text="Note 3.0",
              annotation_position="top",
              annotation_font=dict(size=10, color=GRAY))

# Zone anomalie
fig.add_shape(type="rect",
    x0=3.5, x1=courier_perf["rating"].max() + 0.1,
    y0=27, y1=courier_perf["late_rate"].max() + 2,
    fillcolor=DANGER, opacity=0.05,
    line=dict(color=DANGER, width=1, dash="dot"),
)
fig.add_annotation(
    x=courier_perf["rating"].max(), y=courier_perf["late_rate"].max() + 1,
    text="Zone anomalie RH", showarrow=False,
    font=dict(size=10, color=DANGER), xanchor="right",
)

fig.update_layout(
    title=dict(
        text="<b>Performance Livreurs</b>  ·  Note Client vs Taux de Retard"
             "  ·  Taille = volume livré",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    height=480,
    paper_bgcolor=WHITE,
    plot_bgcolor="#FAFAFA",
    showlegend=False,
    xaxis=dict(title="Note client (/5)", showgrid=True,
               gridcolor="#E2E8F0", zeroline=False, range=[2, 5.2]),
    yaxis=dict(title="Taux de retard (%)", showgrid=True,
               gridcolor="#E2E8F0", zeroline=False),
    margin=dict(t=80, b=60, l=80, r=60),
)
fig.show()

---
## 8. Statuts & Segmentation Clients

Deux visualisations côte à côte : répartition des statuts (donut) + segmentation RFM clients.

In [9]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=(
        "<b>Répartition des statuts</b>",
        "<b>Segmentation clients par récence</b>",
    ),
)

# ── Donut statuts ─────────────────────────────────────────────
status_counts = ov["status_clean"].value_counts()
STATUS_LABELS = {
    "delivered":   "Livré ✅",
    "late":        "Retard ⏰",
    "cancelled":   "Annulé ❌",
    "in_progress": "En cours 🔄",
    "unknown":     "Inconnu",
}
STATUS_COLORS = [SUCCESS, AMBER, DANGER, "#2563EB", GRAY]

fig.add_trace(go.Pie(
    labels=[STATUS_LABELS.get(k, k) for k in status_counts.index],
    values=status_counts.values,
    hole=0.52,
    marker=dict(colors=STATUS_COLORS[:len(status_counts)],
                line=dict(color=WHITE, width=2)),
    hovertemplate="<b>%{label}</b><br>%{value:,} commandes<br>%{percent}<extra></extra>",
    textfont=dict(size=11),
    textinfo="label+percent",
    showlegend=False,
), row=1, col=1)

# Annotation centre
fig.add_annotation(
    text=f"<b>{len(ov):,}</b><br>cmd",
    x=0.22, y=0.5, font=dict(size=13, color=NAVY),
    showarrow=False, xref="paper", yref="paper",
)

# ── Segmentation RFM ─────────────────────────────────────────
last_order = (
    ov.groupby("customer_id")["date_clean"]
    .max()
    .reset_index()
    .rename(columns={"date_clean": "last_order"})
)
REF = pd.Timestamp("2025-03-31")
last_order["days_since"] = (REF - last_order["last_order"]).dt.days
last_order["segment"] = pd.cut(
    last_order["days_since"],
    bins=[0, 30, 90, 180, 99999],
    labels=["Actif 30j", "Actif 90j", "Dormant 180j", "Inactif >180j"],
)
seg_counts = last_order["segment"].value_counts().sort_index()
SEG_COLORS = [SUCCESS, "#34D399", AMBER, DANGER]

fig.add_trace(go.Bar(
    x=seg_counts.index.astype(str),
    y=seg_counts.values,
    marker_color=SEG_COLORS,
    marker_line=dict(color=WHITE, width=1.5),
    text=[f"{v:,}<br>({v/seg_counts.sum()*100:.0f}%)" for v in seg_counts.values],
    textposition="outside",
    textfont=dict(size=11, color=NAVY),
    hovertemplate="<b>%{x}</b><br>%{y:,} clients<extra></extra>",
    showlegend=False,
), row=1, col=2)

# Annotation cible réactivation
fig.add_annotation(
    xref="x2", yref="y2",
    x="Inactif >180j", y=seg_counts["Inactif >180j"] + 30,
    text="💡 Cible réactivation",
    showarrow=True, arrowhead=2, arrowcolor=DANGER,
    font=dict(size=10, color=DANGER),
    ax=0, ay=-40,
)

fig.update_layout(
    title=dict(
        text="<b>Statuts des Commandes & Segmentation Clients (RFM Simplifié)</b>",
        font=dict(size=16, color=NAVY, family="Georgia"), x=0.5,
    ),
    height=440,
    paper_bgcolor=WHITE,
    plot_bgcolor="#FAFAFA",
    margin=dict(t=80, b=60, l=60, r=60),
)
fig.update_yaxes(showgrid=True, gridcolor="#E2E8F0", zeroline=False, row=1, col=2)
fig.update_xaxes(tickangle=-15, row=1, col=2)

fig.show()